<a href="https://colab.research.google.com/github/Obscuriosity/Programming_For_Data_Analysis/blob/main/ST20214513_CMP7005_PRAC_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Programming for Data Analysis CMP7005**

Tony WIllett ST20214513

In [1]:
!pip install streamlit
!pip install --upgrade streamlit

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import streamlit as st
import os
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
import numpy as np

!git config --global user.name "Obscuriosity"
!git config --global user.email "tony.obscuriosity@gmail.com"
username = "Obscuriosity"
repo = "Programming_for_Data_Analysis"
!git clone https://github.com/{username}/{repo}


fatal: destination path 'Programming_for_Data_Analysis' already exists and is not an empty directory.


# Task 1: Data Selection & Handling

You are required to construct a combined dataset for analysis by selecting:

• Select 2 inner (urban) and 2 outer (suburban) stations. provide short justification for your
selection)

▪ Download the four selected datasets.

▪ Import them into your development environment.

▪ Merge them into a single unified dataset for analysis (ensure proper handling of timestamps and station
identifiers).

Choose Locations and upload data.
Monitoring Station:

Dongsi DS - This urban monitoring station seems to be closest to the centre of Beijing

Gucheng GC - This Station is furthest from DS whilst still being an urban station.

Changpingzhen CP - This was chosen because it is an suburban station but is still close to the urban area. It may give information about the influence of the urban area on the local environs.

Huairouzhen HR - This station appears to be the furthest suburban station from the urban centre.

Together these four will give a sort of cross section going form the urban centre, out to the furthest suburbs.


In [3]:
# Load monitoring station data CSV and make pandas dataframes
df_dongsi = pd.read_csv('/content/Programming_for_Data_Analysis/Data/PRSA_Data_Dongsi_20130301-20170228.csv')
df_Gucheng = pd.read_csv('/content/Programming_for_Data_Analysis/Data/PRSA_Data_Gucheng_20130301-20170228.csv') # Corrected file path
df_Changpingzhen = pd.read_csv('/content/Programming_for_Data_Analysis/Data/PRSA_Data_Changping_20130301-20170228.csv')
df_Huairou = pd.read_csv('/content/Programming_for_Data_Analysis/Data/PRSA_Data_Huairou_20130301-20170228.csv')

# Combine dataframes into one
df = pd.concat([df_dongsi, df_Gucheng, df_Changpingzhen, df_Huairou])
df.head()

,No,year,month,day,hour,PM2.5,PM10,SO2,NO2,CO,O3,TEMP,PRES,DEWP,RAIN,wd,WSPM,station
0,1,2013,3,1,0,9.0,9.0,3.0,17.0,300.0,89.0,-0.5,1024.5,-21.4,0.0,NNW,5.7,Dongsi
1,2,2013,3,1,1,4.0,4.0,3.0,16.0,300.0,88.0,-0.7,1025.1,-22.1,0.0,NW,3.9,Dongsi
2,3,2013,3,1,2,7.0,7.0,NaN,17.0,300.0,60.0,-1.2,1025.3,-24.6,0.0,NNW,5.3,Dongsi
3,4,2013,3,1,3,3.0,3.0,5.0,18.0,NaN,NaN,-1.4,1026.2,-25.5,0.0,N,4.9,Dongsi
4,5,2013,3,1,4,3.0,3.0,7.0,NaN,200.0,84.0,-1.9,1027.1,-24.5,0.0,NNW,3.2,Dongsi


# Task 2: Exploratory Data Analysis (EDA)

2.1.Data Understanding
Provide an overview that may include the following, but not limited to:
▪ Number of rows and columns
▪ Column descriptions
▪ Data types
▪ Missing values
▪ Statistical Summary
▪ Initial observations & interpretation

2.2.Data preprocessing:
Perform the necessary data preprocessing steps, including but not limited to handling missing values,
removing duplicate entries, feature engineering (e.g., datetime components, AQI levels), and overall data
cleaning on the main dataset.

2.3.Statistical/Computational Analysis & Visualisation
Perform the necessary steps such as univariate (distribution of pollutants & meteorological variables),
bivariate(e.g. relationships such as PM2.5 vs. Temp, NO2 vs. O3 but not limited to these), and
multivariate analysis (correlation, heatmaps, pairplots), statistical summary, and visualizing the data
(Various charts and graphs, such as bar charts, line charts and scatter plots) that will help in
understanding relationships between variables and to gain important insights from data. Interpret the
key results to demonstrate understanding generated from statistical and visual analysis.

• Explore the dataset however you find meaningful. You may examine different variables, compare
stations, investigate temporal behaviours, or analyse interactions between pollutants and
meteorological factors. Choose the approaches that you believe best help you understand and interpret
the dataset, and present the insights you consider most relevant

In [4]:
# Using Pandas to gain insight into the raw data
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 140256 entries, 0 to 35063
Data columns (total 18 columns):
 #   Column   Non-Null Count   Dtype  
---  ------   --------------   -----  
 0   No       140256 non-null  int64  
 1   year     140256 non-null  int64  
 2   month    140256 non-null  int64  
 3   day      140256 non-null  int64  
 4   hour     140256 non-null  int64  
 5   PM2.5    137133 non-null  float64
 6   PM10     137963 non-null  float64
 7   SO2      137478 non-null  float64
 8   NO2      135681 non-null  float64
 9   CO       132715 non-null  float64
 10  O3       137108 non-null  float64
 11  TEMP     140081 non-null  float64
 12  PRES     140083 non-null  float64
 13  DEWP     140079 non-null  float64
 14  RAIN     140087 non-null  float64
 15  wd       139577 non-null  object 
 16  WSPM     140108 non-null  float64
 17  station  140256 non-null  object 
dtypes: float64(11), int64(5), object(2)
memory usage: 20.3+ MB


The info function shows that there are 140,256 entries and straight away reveals that not all columns contain the full count of data. We can safely assume that the C0 column, which has the fewest non-null entries (132,595) therefore contains the most null data. As would be expected, the date, time and station name data all contin the full count wheras the actual readings all fall short by varying amounts, this may be due to faulty sensors or any number of reasons.

In [5]:
# some pandas functions to elicit individual data insights
print(df.shape)
print(df.columns)
print(df.dtypes)

(140256, 18)
Index(['No', 'year', 'month', 'day', 'hour', 'PM2.5', 'PM10', 'SO2', 'NO2',
       'CO', 'O3', 'TEMP', 'PRES', 'DEWP', 'RAIN', 'wd', 'WSPM', 'station'],
      dtype='object')
No           int64
year         int64
month        int64
day          int64
hour         int64
PM2.5      float64
PM10       float64
SO2        float64
NO2        float64
CO         float64
O3         float64
TEMP       float64
PRES       float64
DEWP       float64
RAIN       float64
wd          object
WSPM       float64
station     object
dtype: object


The shape command confirms that there are 140,256 entries and 18 columns. The columns and dtypes functions repeat the information gained form the info() function, showing the column names and the data types.

In [6]:
# a look at the null values in more detail
print("Missing Valuess")
print(df.isnull().sum())

Missing Valuess
No            0
year          0
month         0
day           0
hour          0
PM2.5      3123
PM10       2293
SO2        2778
NO2        4575
CO         7541
O3         3148
TEMP        175
PRES        173
DEWP        177
RAIN        169
wd          679
WSPM        148
station       0
dtype: int64


The sum of null values command shows that CO indeed contains the most null entries. 7,541 represents just 5.4% of the total 140256 readings. the next highest content of null values is 4,575 for NO2 and equates to 3.3%. This means that the data is quite complete and that imputing values, even if slightly inaccurate will not adversely affect the conclusions drawn. Nonetheless imputation will be done to preserve data integrity as much as possible.

In [9]:
# Create copy for station specific imputation:
df_station_imputed = df.copy()

station-wise imputation: iterate through each unique station in the dataset. For each station:
1.  Filter the DataFrame to include only data for that specific station.
2.  Again dentify numerical columns with missing values within that station's data.
3.  Apply IterativeImputer to fill numerical missing values.
4.  Impute the 'wd' (wind direction) column with the mode specific to that station.
5.  Store the imputed station data.
Finally, all the imputed station dataframes will be combined back into a single DataFrame.

In [10]:
# Creat e four dfs one for aeach station, impute null values.
unique_stations = df_station_imputed['station'].unique()
print(unique_stations)
imputed_dfs_list = []

for station_name in unique_stations:
    print(f"Processing station: {station_name}")
    station_df = df_station_imputed[df_station_imputed['station'] == station_name].copy()

    # Identify numerical columns with missing values for current station
    numerical_cols_with_missing_station = station_df.select_dtypes(include=np.number).columns[station_df.select_dtypes(include=np.number).isnull().any()].tolist()

    if numerical_cols_with_missing_station:
        # Initialize IterativeImputer
        imputer = IterativeImputer(max_iter=10, random_state=0)
        # Apply imputation to columns with missing values for current station
        station_df[numerical_cols_with_missing_station] = imputer.fit_transform(station_df[numerical_cols_with_missing_station])

    # For wind direction, impute with the mode value for current station
    if 'wd' in station_df.columns and station_df['wd'].isnull().any():
        mode_wd_station = station_df['wd'].mode()[0]
        station_df['wd'] = station_df['wd'].fillna(mode_wd_station)

    imputed_dfs_list.append(station_df)

# Join all imputed station dfs back into a single df
df_station_imputed = pd.concat(imputed_dfs_list)

print("\nMissing values after station-wise imputation:")
print(df_station_imputed.isnull().sum())

['Dongsi' 'Gucheng' 'Changping' 'Huairou']
Processing station: Dongsi


/usr/local/lib/python3.12/dist-packages/sklearn/impute/_iterative.py:895: ConvergenceWarning: [IterativeImputer] Early stopping criterion not reached.
  warnings.warn(


Processing station: Gucheng


/usr/local/lib/python3.12/dist-packages/sklearn/impute/_iterative.py:895: ConvergenceWarning: [IterativeImputer] Early stopping criterion not reached.
  warnings.warn(


Processing station: Changping


/usr/local/lib/python3.12/dist-packages/sklearn/impute/_iterative.py:895: ConvergenceWarning: [IterativeImputer] Early stopping criterion not reached.
  warnings.warn(


Processing station: Huairou

Missing values after station-wise imputation:
No         0
year       0
month      0
day        0
hour       0
PM2.5      0
PM10       0
SO2        0
NO2        0
CO         0
O3         0
TEMP       0
PRES       0
DEWP       0
RAIN       0
wd         0
WSPM       0
station    0
dtype: int64


/usr/local/lib/python3.12/dist-packages/sklearn/impute/_iterative.py:895: ConvergenceWarning: [IterativeImputer] Early stopping criterion not reached.
  warnings.warn(


The df_station_imputed DataFrame now contains all the data with missing values imputed on a station by station basis. The output confirms that there are no remaining missing values, ensuring a complete imputed dataset for further analysis.

In [19]:
# Generate a statistical summary of the imputed DataFrame
print("Statistical Summary of the Imputed DataFrame:")
print(df_station_imputed.describe())

Statistical Summary of the Imputed DataFrame:
               No           year          month            day           hour  \
count  140256.000  140256.000000  140256.000000  140256.000000  140256.000000   
mean    17532.500    2014.662560       6.522930      15.729637      11.500000   
std     10122.141       1.177201       3.448715       8.800123       6.922211   
min         1.000    2013.000000       1.000000       1.000000       0.000000   
25%      8766.750    2014.000000       4.000000       8.000000       5.750000   
50%     17532.500    2015.000000       7.000000      16.000000      11.500000   
75%     26298.250    2016.000000      10.000000      23.000000      17.250000   
max     35064.000    2017.000000      12.000000      31.000000      23.000000   

               PM2.5           PM10            SO2            NO2  \
count  140256.000000  140256.000000  140256.000000  140256.000000   
mean       74.498175      97.862905      15.092452      43.651309   
std        75.567

The description provides statistics for each column in the final DataFrame:

*   **count**: Further confirms the number of non-null observations.
*   **mean**: The average value.
*   **std**: The standard deviation.
*   **min**: The minimum value.
*   **25% (Q1)**: The 25th percentile (first quartile).
*   **50% (Median)**: The 50th percentile (median or second quartile).
*   **75% (Q3)**: The 75th percentile (third quartile).
*   **max**: The maximum value.



# Task 3: Model Building:

• After completing all the tasks listed under Task 1 and Task 2, identify and implement the best practices
to build a suitable machine-learning model (e.g., feature scaling, encoding techniques, variable selection,
and parameter optimization).

• Justify your modelling decisions and evaluate model performance using appropriate metrics.


# Task 4: Application Development

Develop an interactive application with a graphical user interface (GUI). The application should include multiple
sections/pages that allow users to explore

• The dataset section,

• Visualization section, and

• Model outputs section.

You may design the structure in any way you find appropriate, but it should enable clear navigation between the

key components of your workflow

# Task 5: Version Control:

• Use GitHub for version control.

• Commit changes regularly with clear, descriptive messages, for example, added PM2.5 prediction
model”, “Created correlation heatmap,” etc.

• Maintain an organised repository structure and include screenshots of:

▫ GitHub commit history

▫ GitHub project repository layout

Links to Papers-
[Yao et al](https://www.mdpi.com/1660-4601/12/10/12264) [Xu&zhang](https://www.sciencedirect.com/science/article/pii/S0301479720301985?via%3Dihub#bib1)